In [ ]:
# --------------------------------------------------------------------------------
# Author        : Arbor Rose
# Edited by     : Jesse Tymas
# Date          : 10.28.25
# Description   : Fetches Domo dataflow previews and flattens:
#                   - Actions (nodes) w/ embedded DAG info (parents/children/degree)
#                   - Tags
#                   - Trigger settings
#                 (Edges are derived on-the-fly; no separate edges dataset.)
# --------------------------------------------------------------------------------
# Revision Date : 10/31/25
# Revision      : Publish Actions-only (with DAG columns), plus Tags & Triggers
# --------------------------------------------------------------------------------

# -----------------------------
# Imports and Setup
# -----------------------------
import os
import json
import sys
import time
import requests
import pandas as pd
import domojupyter as domo
from dotenv import load_dotenv


# -----------------------------
# Configuration
# -----------------------------

ACTIONS_DATASET_NAME           = "Monitoring | Dataflow Actions"                        # FINAL VALUE
TAGS_DATASET_NAME              = "Monitoring | Dataflow Tags"                           # FINAL VALUE
TRIGGERS_DATASET_NAME          = "Monitoring | Dataflow Triggers"                       # FINAL VALUE

REQUEST_TIMEOUT_SEC            = 20                                                     # FINAL VALUE
FETCH_RETRIES                  = 3                                                      # FINAL VALUE
FETCH_BACKOFF_SEC              = 1                                                      # FINAL VALUE

DEBUG                          = False                                                  # FINAL VALUE

load_dotenv()
DOMO_INSTANCE                  = os.getenv("DOMO_INSTANCE")
DOMO_DEV_TOKEN                 = os.getenv("DOMO_DEV_TOKEN")


# -----------------------------
# Helper: nested search for a key
# -----------------------------
def find_key(d, key):
    if isinstance(d, dict):
        for k, v in d.items():
            if k == key:
                return v
            res = find_key(v, key)
            if res is not None:
                return res
    elif isinstance(d, list):
        for item in d:
            res = find_key(item, key)
            if res is not None:
                return res
    return None


# -----------------------------
# Helper: list all dataflow IDs
# -----------------------------
def list_dataflow_ids(instance, dev_token):
    url = f"https://{instance}.domo.com/api/dataprocessing/v1/dataflows"
    headers = {"x-domo-developer-token": dev_token, "Accept": "application/json"}
    resp = requests.get(url, headers=headers)
    resp.raise_for_status()
    batch = resp.json()
    return [df.get("id") for df in batch if "id" in df]


# -----------------------------
# Helper: fetch dataflow preview
# -----------------------------
def fetch_dataflow_preview(instance, df_id, dev_token, retries=FETCH_RETRIES, backoff=FETCH_BACKOFF_SEC):
    url = f"https://{instance}.domo.com/api/dataprocessing/v2/dataflows/{df_id}"
    headers = {"x-domo-developer-token": dev_token, "Accept": "application/json"}
    params = {"validationType": "PREVIEW"}

    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(url, headers=headers, params=params, timeout=REQUEST_TIMEOUT_SEC)
            if resp.status_code == 200:
                return resp.json()
            else:
                print(f"[Attempt {attempt}] HTTP {resp.status_code} for {df_id}")
        except requests.RequestException as e:
            print(f"[Attempt {attempt}] Request error for {df_id}: {e}")
        time.sleep(backoff * attempt)

    return None


# -----------------------------
# Small helpers for extractors
# -----------------------------
def safe_name(preview_json):
    return preview_json.get("displayName") or preview_json.get("name") or "<no name>"

def ensure_preview_id(preview_json, df_id):
    preview_json.setdefault("id", df_id)
    return preview_json


# -----------------------------
# Extractor: Actions (nodes)
# -----------------------------
def extract_actions_rows(preview_json):
    """
    One row per action node. Includes common properties, GUI coords, and a few
    action-specific flags seen in payloads. Keeps a raw JSON column for review.
    DAG columns (parents/children/degree) are added later in a post-process step.
    """
    rows = []
    df_id = preview_json.get("id")
    name = safe_name(preview_json)

    for a in (preview_json.get("actions") or []):
        rows.append({
            "dataflow_id": df_id,
            "dataflow_name": name,
            "action_id": a.get("id"),
            "action_type": a.get("type"),
            "action_name": a.get("name"),
            "depends_on": ",".join(a.get("dependsOn", []) or []),
            "input_single": a.get("input"),
            "inputs_list": ",".join(a.get("inputs", []) or []),
            "settings_preferredDatabaseEntityType": (a.get("settings") or {}).get("preferredDatabaseEntityType"),
            "propagateAi": a.get("propagateAi"),
            "executeFlowWhenUpdated": a.get("executeFlowWhenUpdated"),
            "pseudoDataSource": a.get("pseudoDataSource"),
            "truncateTextColumns": a.get("truncateTextColumns"),
            "truncateRows": a.get("truncateRows"),
            "onlyLoadNewVersions": a.get("onlyLoadNewVersions"),
            "recentVersionCutoffMs": a.get("recentVersionCutoffMs"),
            "dataSourceId": a.get("dataSourceId"),
            "gui_x": (a.get("gui") or {}).get("x"),
            "gui_y": (a.get("gui") or {}).get("y"),
            "action_raw": json.dumps(a),
        })
    return rows


# -----------------------------
# Post-process: enrich Actions with DAG columns
# -----------------------------
def _csv_to_set(s):
    if not s:
        return set()
    return {x for x in (str(s).split(",")) if x}

def _to_csv(values):
    return ",".join(sorted(str(v) for v in (values or []) if v))

def add_dag_columns_to_actions(actions_df: pd.DataFrame) -> pd.DataFrame:
    """
    Build edges from the three wiring fields in actions_df (depends_on, input_single, inputs_list)
    and enrich actions_df with parents/children lists and degree counts.
    """
    if actions_df.empty:
        for col in [
            "parents_dependsOn","parents_input","parents_inputsList","parents_all",
            "children_dependsOn","children_input","children_inputsList","children_all",
            "in_degree","out_degree",
        ]:
            actions_df[col] = "" if "parents" in col or "children" in col else 0
        return actions_df

    # Construct edge lists from action wiring
    edges = []  # tuples: (edge_type, from, to)
    for _, row in actions_df.iterrows():
        to_id = row.get("action_id")

        for src in _csv_to_set(row.get("depends_on")):
            edges.append(("dependsOn", src, to_id))

        inp = row.get("input_single")
        if inp:
            edges.append(("input", str(inp), to_id))

        for src in _csv_to_set(row.get("inputs_list")):
            edges.append(("inputs", src, to_id))

    # Build parent/child maps
    parents_dep, parents_inp, parents_inps = {}, {}, {}
    children_dep, children_inp, children_inps = {}, {}, {}

    for et, src, dst in edges:
        if et == "dependsOn":
            parents_dep.setdefault(dst, set()).add(src)
            children_dep.setdefault(src, set()).add(dst)
        elif et == "input":
            parents_inp.setdefault(dst, set()).add(src)
            children_inp.setdefault(src, set()).add(dst)
        elif et == "inputs":
            parents_inps.setdefault(dst, set()).add(src)
            children_inps.setdefault(src, set()).add(dst)

    def parents_all(aid):
        return parents_dep.get(aid, set()) | parents_inp.get(aid, set()) | parents_inps.get(aid, set())

    def children_all(aid):
        return children_dep.get(aid, set()) | children_inp.get(aid, set()) | children_inps.get(aid, set())

    out = actions_df.copy()

    # Parents
    out["parents_dependsOn"]   = out["action_id"].map(lambda a: _to_csv(parents_dep.get(a, set())))
    out["parents_input"]       = out["action_id"].map(lambda a: _to_csv(parents_inp.get(a, set())))
    out["parents_inputsList"]  = out["action_id"].map(lambda a: _to_csv(parents_inps.get(a, set())))
    out["parents_all"]         = out["action_id"].map(lambda a: _to_csv(parents_all(a)))

    # Children
    out["children_dependsOn"]  = out["action_id"].map(lambda a: _to_csv(children_dep.get(a, set())))
    out["children_input"]      = out["action_id"].map(lambda a: _to_csv(children_inp.get(a, set())))
    out["children_inputsList"] = out["action_id"].map(lambda a: _to_csv(children_inps.get(a, set())))
    out["children_all"]        = out["action_id"].map(lambda a: _to_csv(children_all(a)))

    # Degrees
    out["in_degree"]  = out["action_id"].map(lambda a: len(parents_all(a)))
    out["out_degree"] = out["action_id"].map(lambda a: len(children_all(a)))

    return out


# -----------------------------
# Extractor: Tags
# -----------------------------
def extract_tags_rows(preview_json):
    rows = []
    df_id = preview_json.get("id")
    name = safe_name(preview_json)

    tags = preview_json.get("tags")
    if tags is None:
        tags = find_key(preview_json, "tags")
    if not tags:
        return rows

    tags_list = tags if isinstance(tags, list) else tags.get("tags", [])
    for tag in (tags_list or []):
        if isinstance(tag, dict):
            rows.append({
                "dataflow_id": df_id,
                "dataflow_name": name,
                "tag_name": tag.get("name") or tag.get("value") or tag.get("label"),
            })
        else:
            rows.append({
                "dataflow_id": df_id,
                "dataflow_name": name,
                "tag_name": str(tag),
                "tag_raw": json.dumps(tag)
            })
    return rows


# -----------------------------
# Extractor: Triggers
# -----------------------------
def extract_triggers_rows(preview_json):
    rows = []
    df_id = preview_json.get("id")
    name = safe_name(preview_json)

    ts = preview_json.get("triggerSettings") or find_key(preview_json, "triggerSettings")
    if not ts:
        return rows

    zoneId = ts.get("zoneId")
    locale = ts.get("locale")

    for trig in (ts.get("triggers") or []):
        trigger_title = trig.get("title")
        trigger_id = trig.get("triggerId")
        events = trig.get("triggerEvents") or []

        for evt in events:
            sched = evt.get("schedule", {}) or {}
            rows.append({
                "dataflow_id": df_id,
                "dataflow_name": name,
                "trigger_title": trigger_title,
                "trigger_id": trigger_id,
                "trigger_type": evt.get("type"),
                "datasetId": evt.get("datasetId") or evt.get("id"),
                "triggerOnDataChanged": evt.get("triggerOnDataChanged"),
                "schedule_second": sched.get("second"),
                "schedule_minute": sched.get("minute"),
                "schedule_hour": sched.get("hour"),
                "schedule_dayOfMonth": sched.get("dayOfMonth"),
                "schedule_month": sched.get("month"),
                "schedule_dayOfWeek": sched.get("dayOfWeek"),
                "schedule_year": sched.get("year"),
                "zoneId": zoneId,
                "locale": locale
            })

    return rows


# -----------------------------
# Main
# -----------------------------
def main():
    # Discover dataflows
    dataflow_ids = list_dataflow_ids(DOMO_INSTANCE, DOMO_DEV_TOKEN)
    print(f"Found {len(dataflow_ids)} dataflow IDs.\n")
    sys.stdout.flush()

    actions_out = []
    tags_out = []
    triggers_out = []

    # Fetch once per dataflow, fan out to extractors
    for idx, df_id in enumerate(dataflow_ids, start=1):
        if DEBUG:
            print(f"[{idx}/{len(dataflow_ids)}] Fetching {df_id}")
        sys.stdout.flush()

        preview = fetch_dataflow_preview(DOMO_INSTANCE, df_id, DOMO_DEV_TOKEN)
        if not preview:
            print(f"  Failed to fetch {df_id}")
            continue

        ensure_preview_id(preview, df_id)

        try:
            actions_out.extend(extract_actions_rows(preview))
            tags_out.extend(extract_tags_rows(preview))
            triggers_out.extend(extract_triggers_rows(preview))
            if DEBUG:
                print("  Extracted nodes, tags, triggers.")
        except Exception as e:
            print(f"  Skipping extractors for {df_id}: {e}")

    # Build DataFrames
    actions_df  = pd.DataFrame(actions_out)
    tags_df     = pd.DataFrame(tags_out)
    triggers_df = pd.DataFrame(triggers_out)

    # Enrich actions with DAG columns (parents/children/degree)
    actions_df = add_dag_columns_to_actions(actions_df)

    print(f"\n✅ Actions rows:  {len(actions_df)}")
    print(f"✅ Tags rows:     {len(tags_df)}")
    print(f"✅ Triggers rows: {len(triggers_df)}\n")

    domo.write_dataframe(actions_df,  ACTIONS_DATASET_NAME)
    domo.write_dataframe(tags_df,     TAGS_DATASET_NAME)
    domo.write_dataframe(triggers_df, TRIGGERS_DATASET_NAME)
    
    return actions_df, tags_df, triggers_df


In [ ]:
actions_df, tags_df, triggers_df = main()